# Preparar dados para exploração e modelagem

Este notebook baixa os arquivos mensais do VRA e gera as bases locais usadas pelos notebooks `01_entendimento_do_dataset.ipynb` e `02_modelagem_faixas_atraso.ipynb`.

Para datasets oficiais, imutáveis e vinculados a runs, use o materializador em `datasets/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts').exists():
    raise FileNotFoundError('Não encontrei a raiz do projeto com a pasta scripts/.')

INICIO_MES = '2024-01'
FIM_MES = '2025-12'
print('Raiz:', ROOT)
print('Período:', INICIO_MES, 'até', FIM_MES)

## Limpar arquivos brutos anteriores

Antes de baixar, removemos somente os CSVs VRA e arquivos temporários gerados anteriormente em `data/raw/`. Arquivos com outros nomes não são alterados.

In [ ]:
RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
arquivos_anteriores = sorted(list(RAW_DIR.glob('VRA_*.csv')) + list(RAW_DIR.glob('VRA_*.csv.download')))
for arquivo in arquivos_anteriores:
    arquivo.unlink()
    print('Removido:', arquivo)
print(f'{len(arquivos_anteriores)} arquivo(s) anterior(es) removido(s).')

## Baixar os arquivos mensais

O intervalo é inclusivo. O downloader informa o início e o sucesso de cada mês no output da célula.

In [ ]:
def executar(*argumentos):
    comando = [sys.executable, *argumentos]
    print('+', ' '.join(comando))
    subprocess.run(comando, cwd=ROOT, check=True)

executar(
    'scripts/baixar_amostra.py',
    '--inicio-mes', INICIO_MES,
    '--fim-mes', FIM_MES,
    '--output-dir', 'data/raw',
    '--metadata-output', 'data/amostra_metadata.json',
    '--sobrescrever',
)

## Auditar os arquivos brutos

A auditoria verifica a estrutura dos CSVs baixados, incluindo colunas, duplicatas e valores ausentes. Ela gera `data/auditoria_amostra.json`.

In [ ]:
executar(
    'scripts/auditar_amostra_v2.py',
    '--input-dir', 'data/raw',
    '--output', 'data/auditoria_amostra.json',
)

## Gerar o dataset derivado

Esta etapa normaliza os nomes, datas e tipos dos arquivos brutos e gera `data/voos_vra_derivados.csv`, que é usado pelo notebook de entendimento. O script:

- lê todos os arquivos `VRA_*.csv` encontrados em `data/raw/`;
- valida e seleciona as colunas esperadas do VRA e renomeia os campos para nomes padronizados;
- converte as quatro colunas de data e hora para tipos de data do pandas;
- calcula os atrasos de partida e chegada em minutos;
- cria os indicadores booleanos de voo realizado e cancelado;
- mantém os atrasos em minutos ausentes quando os horários necessários não estão disponíveis;
- registra o arquivo e a linha de origem para rastreabilidade;
- concatena os meses em uma única tabela e salva o CSV derivado.

O script não cria ainda a coluna `faixa_atraso`; essa materialização ocorre na etapa seguinte, ao gerar a base de modelagem.

In [ ]:
executar(
    'scripts/preparar_dados_v2.py',
    '--input-dir', 'data/raw',
    '--output', 'data/voos_vra_derivados.csv',
)

## Materializar a base de modelagem e as faixas de atraso

Esta etapa lê o dataset derivado, mantém somente os voos realizados com atraso de chegada calculável e com antecipação não inferior a 24 horas, cria a coluna-alvo `faixa_atraso`, gera as variáveis temporais e salva `data/modelagem_faixas_atraso.csv`, que será usado pelos modelos. O output informa as linhas não elegíveis, detalha quantas foram removidas por cada motivo, registra os critérios aplicados, a redução líquida de colunas, as colunas brutas excluídas e as colunas criadas. Também gera a distribuição mensal e exibe a quantidade e o percentual de registros em cada uma das seis classes.

In [ ]:
executar(
    'scripts/preparar_modelagem.py',
    '--input', 'data/voos_vra_derivados.csv',
    '--output', 'data/modelagem_faixas_atraso.csv',
    '--monthly-output', 'data/distribuicao_mensal_faixas_atraso.csv',
)

## Preparar a conferência

Primeiro importamos o pandas e registramos os nomes das seis classes para que a tabela final seja apresentada em ordem e com descrições legíveis.

In [ ]:
import pandas as pd

FAIXAS = {
    0: 'Pontual ou antecipado',
    1: 'Atraso inferior a 15 min',
    2: 'Atraso de 15 a 30 min',
    3: 'Atraso superior a 30 até 45 min',
    4: 'Atraso superior a 45 até 60 min',
    5: 'Atraso superior a 60 min',
}

## Carregar e conferir as bases

Agora carregamos o dataset derivado e a base de modelagem e mostramos suas dimensões. Isso confirma que os arquivos esperados foram gerados antes de analisar as classes.

In [ ]:
derivado = pd.read_csv(ROOT / 'data' / 'voos_vra_derivados.csv', low_memory=False)
modelagem = pd.read_csv(ROOT / 'data' / 'modelagem_faixas_atraso.csv', low_memory=False)
print('data/voos_vra_derivados.csv:', derivado.shape)
print('data/modelagem_faixas_atraso.csv:', modelagem.shape)

## Estatísticas numéricas - dados originais

As chamadas `describe()` abaixo resumem, por padrão, somente as colunas numéricas. 
Por isso o dataset derivado mostra 
- `numero_assentos`,
- `atraso_partida_min`
- `atraso_chegada_min`
- `linha_origem`


In [53]:
derivado.describe().style.format('{:,.2f}')

,numero_assentos,atraso_partida_min,atraso_chegada_min,linha_origem
count,"1,992,832.00","1,864,195.00","1,864,195.00","1,992,832.00"
mean,161.91,7.62,3.85,"41,588.70"
std,69.14,74.59,75.60,"24,093.09"
min,0.00,"-5,760.00","-5,784.00",1.00
25%,136.00,-7.00,-12.00,"20,759.00"
50%,180.00,-1.00,-4.00,"41,518.00"
75%,186.00,9.00,8.00,"62,276.25"
max,515.00,"44,635.00","44,625.00","89,616.00"


In [49]:
derivado.describe()

,numero_assentos,atraso_partida_min,atraso_chegada_min,linha_origem
count,1.992832e+06,1.864195e+06,1.864195e+06,1.992832e+06
mean,1.619054e+02,7.621729e+00,3.851277e+00,4.158870e+04
std,6.914173e+01,7.458818e+01,7.559674e+01,2.409309e+04
min,0.000000e+00,-5.760000e+03,-5.784000e+03,1.000000e+00
25%,1.360000e+02,-7.000000e+00,-1.200000e+01,2.075900e+04
50%,1.800000e+02,-1.000000e+00,-4.000000e+00,4.151800e+04
75%,1.860000e+02,9.000000e+00,8.000000e+00,6.227625e+04
max,5.150000e+02,4.463500e+04,4.462500e+04,8.961600e+04


## Estatísticas numéricas - Após modelagem

As chamadas `describe()` abaixo resumem, por padrão, somente as colunas numéricas:
a base de modelagem mostra 
- numero_assentos
- variáveis de calendário
- fim_de_semana
- o código ordinal `faixa_atraso`.

Nem todas essas colunas representam medidas contínuas: `ano`, `mes`, `dia_semana`, `hora_prevista`, `fim_de_semana` e `faixa_atraso` são variáveis de calendário, indicadores ou códigos. Portanto, suas médias e desvios servem apenas como inspeção técnica; a distribuição das faixas deve ser consultada na tabela específica abaixo.

In [50]:
modelagem.head()

,id_registro,data_referencia,mes_referencia,companhia_icao,origem_icao,destino_icao,codigo_tipo_linha,modelo_equipamento,numero_assentos,ano,mes,dia_semana,hora_prevista,fim_de_semana,periodo_dia,faixa_atraso
0,VRA_2024_01.csv:1,2024-01-01,2024-01,AAL,SBGL,KMIA,I,B772,288,2024,1,0,23,0,noite,3
1,VRA_2024_01.csv:62044,2024-01-01,2024-01,TAM,SBPA,SBGR,N,A320,180,2024,1,0,6,0,manha,0
2,VRA_2024_01.csv:62043,2024-01-01,2024-01,TAM,SBGR,SBBE,N,A321,224,2024,1,0,7,0,manha,1
3,VRA_2024_01.csv:62042,2024-01-01,2024-01,TAM,SBJV,SBGR,N,A320,180,2024,1,0,10,0,manha,0
4,VRA_2024_01.csv:62041,2024-01-01,2024-01,TAM,SBGO,SBGR,N,A321,224,2024,1,0,14,0,tarde,0


In [52]:
modelagem.describe().style.format('{:,.2f}')

,numero_assentos,ano,mes,dia_semana,hora_prevista,fim_de_semana,faixa_atraso
count,"1,863,908.00","1,863,908.00","1,863,908.00","1,863,908.00","1,863,908.00","1,863,908.00","1,863,908.00"
mean,166.02,"2,024.51",6.54,2.94,12.86,0.27,0.74
std,64.63,0.50,3.46,1.99,5.85,0.44,1.25
min,0.00,"2,024.00",1.00,0.00,0.00,0.00,0.00
25%,136.00,"2,024.00",4.00,1.00,8.00,0.00,0.00
50%,180.00,"2,025.00",7.00,3.00,13.00,0.00,0.00
75%,186.00,"2,025.00",10.00,5.00,18.00,1.00,1.00
max,515.00,"2,025.00",12.00,6.00,23.00,1.00,5.00


## Distribuição das faixas de atraso

Por fim, contamos os registros de cada `faixa_atraso` e calculamos seu percentual na base de modelagem. Essa tabela permite verificar o desbalanceamento entre as classes antes do treinamento.

In [ ]:
distribuicao = (
    modelagem['faixa_atraso'].value_counts()
    .reindex(FAIXAS.keys(), fill_value=0)
    .rename('quantidade')
    .to_frame()
)
distribuicao.index.name = 'codigo'
distribuicao['classificacao'] = distribuicao.index.map(FAIXAS)
distribuicao['percentual'] = (100 * distribuicao['quantidade'] / len(modelagem)).round(2)
distribuicao[['classificacao', 'quantidade', 'percentual']]